<a href="https://colab.research.google.com/github/zty2020china/IOS13-SimulateTouch/blob/master/%E7%84%A6%E7%82%89%E7%83%AD%E5%B7%A5%E4%BB%BF%E7%9C%9F%E5%B9%B3%E5%8F%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
焦炉专业封装软件 V1.6 - 饱和温湿联动集成版
1. 解决 __file__ 在 Notebook 中的报错问题
2. 集成用户提供的饱和水汽体积数据，并自动换算为质量 (g)
3. 联动炉型数据库，一键生成单室废气量报告
"""

import os
import math
import gradio as gr
import pandas as pd
from pydantic import BaseModel

# Mac 环境网络补丁
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["NO_PROXY"] = "localhost,127.0.0.1,0.0.0.0"

# ==========================================
# 1. 核心物理数据与转换逻辑
# ==========================================
WATER_VAPOR_DENSITY = 804.0  # 标况下水蒸气密度 g/Nm3

# 用户提供的饱和水汽数据 (温度: m3水汽/Nm3干气)
SAT_WATER_VOL_DATA = {
    20: 0.0235, 25: 0.0323, 30: 0.0436,
    35: 0.0587, 40: 0.0785, 45: 0.1043
}

def get_water_mass_by_temp(temp_c):
    """根据温度计算饱和水质量 (g/Nm3)"""
    # 采用指数拟合公式以适配任意温度输入 (基于用户数据点拟合: y = 0.0084 * e^(0.0558 * x))
    vol_m3 = 0.0084 * math.exp(0.0558 * temp_c)
    return vol_m3 * WATER_VAPOR_DENSITY

# ==========================================
# 2. 数据库引擎 (含路径修复补丁)
# ==========================================
class OvenDatabase:
    def __init__(self, csv_name="焦炉常用参数 - 工作表1.csv"):
        # 路径修复补丁：兼容 .py 脚本和 .ipynb 笔记本
        try:
            base_path = os.path.dirname(os.path.abspath(__file__))
        except NameError:
            base_path = os.getcwd() # 在 Notebook 中使用当前工作目录

        self.csv_path = os.path.join(base_path, csv_name)
        self.db = {}
        self.load_from_csv()

    def load_from_csv(self):
        if not os.path.exists(self.csv_path):
            print(f"⚠️ 找不到文件，路径是：{self.csv_path}\n将使用内置演示数据。")
            self.db = {
                "7.63m 示例": {"每孔炭化室装煤量（t）": "57.3", "周转时间": "25", "成焦率": "75", "炭化室长（mm）": "18560"},
                "JN43 示例": {"每孔炭化室装煤量（t）": "17.9", "周转时间": "18", "成焦率": "73", "炭化室长（mm）": "14080"}
            }
            return

        for enc in ['gbk', 'utf-8-sig', 'gb2312']:
            try:
                df = pd.read_csv(self.csv_path, encoding=enc, index_col=0)
                df_t = df.T
                for model_name, row_data in df_t.iterrows():
                    self.db[str(model_name).strip()] = {k.strip(): str(v) for k, v in row_data.items() if pd.notna(v)}
                print(f"✅ 成功加载 {len(self.db)} 种炉型数据")
                break
            except: continue

    def get_models(self):
        return list(self.db.keys())

    def get_params(self, name):
        return self.db.get(name, {})

oven_db = OvenDatabase()

# ==========================================
# 3. 计算引擎 (燃烧与热平衡)
# ==========================================
class CokeOvenEngine:
    @staticmethod
    def run_simulation(fuel_temp, fuel_hum, air_temp, air_rh, alpha, coal_mt, oven_schedule):
        # 1. 燃烧基础 (假设焦炉煤气典型组分)
        # H2:58, CH4:25, CO:6, CmHn:2.5, N2:5.0, CO2:2.5, H2S:0.5, O2:0.5
        lhv = 17500.0 # kJ/Nm3 (典型值)
        v0_air = 4.25  # 理论空气量

        # 2. 湿度修正
        p_sat_air = 6.112 * math.exp((17.62 * air_temp) / (243.12 + air_temp))
        x_w_air = (air_rh / 100.0) * p_sat_air / 1013.25
        v_act_air = (alpha * v0_air) / (1 - x_w_air)

        # 废气量计算 (包含燃料水、生成水、空气水)
        # 燃料带入水蒸气体积 (用户数据核心应用)
        v_h2o_fuel = fuel_hum / WATER_VAPOR_DENSITY
        v_act_flue = 5.0 + (alpha-1)*v0_air + v_h2o_fuel + (v_act_air * x_w_air)

        # 3. 产能与单室废气
        dry_coal = oven_schedule['charge'] * (1 - coal_mt/100.0)
        annual_cap = (oven_schedule['count'] / oven_schedule['time']) * 8000 * oven_schedule['charge'] * (oven_schedule['rate']/100.0)

        # 耗热量估算 (q_dry)
        q_dry = 2350 + (coal_mt * 25) # 随水分波动的简化耗热量公式
        hourly_dry_kg = (oven_schedule['count'] / oven_schedule['time']) * dry_coal * 1000.0
        total_flue_h = (hourly_dry_kg * q_dry / lhv) * v_act_flue

        return {
            "capacity": annual_cap,
            "metrics": {
                "燃料入炉含水质量": f"{fuel_hum:.2f} g/Nm³ (根据{fuel_temp}℃饱和点自动计算)",
                "综合干煤耗热量": f"{q_dry:.0f} kJ/kg",
                "全炉实际废气量": f"{total_flue_h:.0f} Nm³/h",
                "单标准燃烧室废气量": f"**{total_flue_h / oven_schedule['count']:.0f} Nm³/h**"
            }
        }

# ==========================================
# 4. Gradio 交互界面
# ==========================================
def on_temp_change(temp, auto_mode):
    if auto_mode:
        return round(get_water_mass_by_temp(temp), 2)
    return gr.update()

def on_model_change(name):
    d = oven_db.get_params(name)
    info = f"**物理尺寸**: 长{d.get('炭化室长（mm）','-')}mm | 高{d.get('炭化室高（mm）','-')}mm"
    return float(d.get("每孔炭化室装煤量（t）", 30)), float(d.get("周转时间", 20)), float(d.get("成焦率", 75)), info

def main_calc(f_temp, f_hum, a_temp, a_rh, alpha, c_mt, o_cnt, o_chg, o_time, o_rate):
    sched = {'count': o_cnt, 'charge': o_chg, 'time': o_time, 'rate': o_rate}
    res = CokeOvenEngine.run_simulation(f_temp, f_hum, a_temp, a_rh, alpha, c_mt, sched)

    report = f"### 📊 焦炉热工集成计算书 (V1.6)\n\n"
    report += f"**项目单年标定产能：{res['capacity']/10000:.2f} 万吨/年** (8000h工制)\n\n"
    report += "| 参数名称 | 计算结果 |\n| :--- | :--- |\n"
    for k, v in res["metrics"].items():
        report += f"| {k} | {v} |\n"
    return report

with gr.Blocks(title="焦炉综合仿真 V1.6") as app:
    gr.Markdown("# 🏭 焦炉热工仿真平台 V1.6")
    gr.Markdown("集成饱和温湿联动逻辑：根据燃料温度自动带出水分质量浓度 ($g/Nm^3$)")

    with gr.Row():
        with gr.Column():
            with gr.Group():
                gr.Markdown("### 📚 炉型数据库")
                model_dd = gr.Dropdown(choices=oven_db.get_models(), label="选择炉型", value=oven_db.get_models()[0] if oven_db.get_models() else None)
                model_info = gr.Markdown("-")
                o_cnt = gr.Number(65, label="焦炉孔数")
                o_chg = gr.Number(label="单孔装煤量(t)")
                o_time = gr.Number(label="周转时间(h)")
                o_rate = gr.Number(label="成焦率(%)")

            with gr.Group():
                gr.Markdown("### 💧 燃料水分联动")
                f_temp = gr.Slider(20, 45, value=35, label="燃料煤气温度 (℃)")
                auto_mode = gr.Checkbox(True, label="自动同步饱和含水量 (根据实测数据)")
                f_hum = gr.Number(47.2, label="煤气含湿量 (g/Nm³)")

        with gr.Column():
            with gr.Group():
                gr.Markdown("### ⚙️ 工况与煤质")
                alpha = gr.Slider(1.1, 1.4, 1.2, label="空气过剩系数")
                c_mt = gr.Slider(5, 15, 10, label="入炉煤水分 %")
                a_temp = gr.Slider(0, 40, 25, label="环境气温 (℃)")
                a_rh = gr.Slider(0, 100, 60, label="空气相对湿度 %")

            btn = gr.Button("🚀 开始综合仿真", variant="primary")
            res_md = gr.Markdown("### 📑 计算报告\n等待执行...")

    # 事件绑定
    model_dd.change(on_model_change, model_dd, [o_chg, o_time, o_rate, model_info])
    f_temp.change(on_temp_change, [f_temp, auto_mode], f_hum)
    btn.click(main_calc, [f_temp, f_hum, a_temp, a_rh, alpha, c_mt, o_cnt, o_chg, o_time, o_rate], res_md)
    app.load(on_model_change, model_dd, [o_chg, o_time, o_rate, model_info])

if __name__ == "__main__":
    app.launch(server_name="127.0.0.1", server_port=7860)